# MNIST MLP3: SGD + momentum versus the original TraceLogRG

This notebook is in `optimizers/trace_log_tracker/` and tests the **first paper-based trace-log optimizer**, not the later self-consistent ECS or spectral-flow algorithms.

The original WeightWatcher full-$M$ boundaries define

$$
m_R=\left\lfloor\frac{m_{\mathrm{PL}}+m_{\mathrm{TL}}}{2}\right\rfloor,
$$

with $m_{\mathrm{PL}}=\texttt{num\_pl\_spikes}$ and $m_{\mathrm{TL}}=\texttt{detX\_num}$. For each completed SGD-plus-momentum displacement,

$$
d=\langle G_T,\Delta W\rangle_F,
\qquad
a^-=\min\left(\frac{d}{\lVert G_T\rVert_F^2},0\right),
$$

$$
\Delta W_{\not\rightarrow F_0}=\Delta W-a^-G_T.
$$

The baseline and wrapped models start from identical weights and consume the same minibatches. The base optimizer is ordinary `torch.optim.SGD(lr=0.05, momentum=0.9, nesterov=False)`. There is no Muon or Newton--Schulz step. The one-sided correction runs every minibatch on all eligible layers at full strength with no correction cap.

In [ ]:
from pathlib import Path
import copy, json, math, random, sys
import matplotlib.pyplot as plt
import numpy as np, pandas as pd, torch, torch.nn.functional as F
import weightwatcher as ww
from IPython.display import display
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

ROOT = None
for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / "rg_trace_log").is_dir():
        ROOT = p
        break
    q = p / "optimizers" / "trace_log_tracker"
    if (q / "rg_trace_log").is_dir():
        ROOT = q
        break
if ROOT is None:
    raise RuntimeError("Run this notebook from a clone of CalculatedContent/rg_optimizers.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from rg_trace_log import TraceLogConfig, TraceLogRGWrapper
from rg_trace_log.mnist_experiment import (
    MLP3, MNISTExperimentConfig, choose_device, evaluate, set_seed,
    _measure, _train_pair_one_epoch, _summarize_corrections,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 300)

cfg = MNISTExperimentConfig(
    seed=1337, epochs=20, batch_size=128, learning_rate=0.05,
    weight_decay=1e-4, grad_clip_norm=1.0,
    rg_mode="one_sided", rg_normalization="weightwatcher",
    rg_gamma=0.10, rg_ridge_relative=1e-6, rg_min_retained=5,
    rg_correction_scale=1.0, rg_max_correction_ratio=None,
    rg_apply_every_steps=1, rg_warmup_steps=0,
    ww_min_evals=10, ww_max_evals=None, n_log_shells=5,
    min_retained_for_beta=20, min_decades_for_beta=0.50,
    train_eval_max_batches=50,
)
MOMENTUM, DAMPENING, NESTEROV = 0.9, 0.0, False
assert cfg.rg_mode == "one_sided" and cfg.rg_apply_every_steps == 1
assert cfg.rg_max_correction_ratio is None and cfg.rg_normalization == "weightwatcher"
print("root:", ROOT, "| torch:", torch.__version__, "| WW:", getattr(ww, "__version__", "unknown"))

In [ ]:
set_seed(cfg.seed)
device = choose_device()
tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_ds = datasets.MNIST(str(ROOT / "data"), train=True, download=True, transform=tf)
test_ds = datasets.MNIST(str(ROOT / "data"), train=False, download=True, transform=tf)
gen = torch.Generator().manual_seed(cfg.seed)
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, generator=gen, num_workers=0)
train_eval = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=0)

state = copy.deepcopy(MLP3().state_dict())
base_model = MLP3().to(device); base_model.load_state_dict(state)
rg_model = MLP3().to(device); rg_model.load_state_dict(state)
def make_sgd(model):
    return torch.optim.SGD(
        model.parameters(), lr=cfg.learning_rate, momentum=MOMENTUM,
        dampening=DAMPENING, weight_decay=cfg.weight_decay, nesterov=NESTEROV,
    )
base_opt, rg_base = make_sgd(base_model), make_sgd(rg_model)
rg_opt = TraceLogRGWrapper(
    rg_base, rg_model.named_parameters(),
    config=TraceLogConfig(
        mode=cfg.rg_mode, gamma=cfg.rg_gamma, normalization=cfg.rg_normalization,
        ridge_relative=cfg.rg_ridge_relative, min_retained=cfg.rg_min_retained,
        correction_scale=cfg.rg_correction_scale,
        max_correction_ratio=cfg.rg_max_correction_ratio,
        apply_every_steps=cfg.rg_apply_every_steps, warmup_steps=cfg.rg_warmup_steps,
    ),
)
assert base_opt.__class__ is torch.optim.SGD and rg_base.__class__ is torch.optim.SGD
assert base_opt.param_groups[0]["momentum"] == MOMENTUM
BASE, RG = "SGD + momentum baseline", "SGD + momentum + TraceLogRG"

initial_test = evaluate(rg_model, test_loader, device=device)
initial_rg = _measure(rg_model, run_label=RG, epoch=0, global_step=0, config=cfg)
rg_opt.set_supports(initial_rg.supports)
initial_base = initial_rg.metrics.copy()
if not initial_base.empty:
    initial_base["run"] = BASE

perf = [
    dict(epoch=0, run=BASE, train_loss=np.nan, train_acc=np.nan,
         test_loss=initial_test["loss"], test_acc=initial_test["acc"]),
    dict(epoch=0, run=RG, train_loss=np.nan, train_acc=np.nan,
         test_loss=initial_test["loss"], test_acc=initial_test["acc"]),
]
ww_frames, step_frames, global_step = [initial_base, initial_rg.metrics.copy()], [], 0

for epoch in range(1, cfg.epochs + 1):
    steps = _train_pair_one_epoch(
        base_model, rg_model, base_opt, rg_opt, train_loader,
        epoch=epoch, device=device, grad_clip_norm=cfg.grad_clip_norm,
    )
    if not steps.empty:
        step_frames.append(steps)
    global_step += len(train_loader)
    bt = evaluate(base_model, train_eval, device=device, max_batches=cfg.train_eval_max_batches)
    bv = evaluate(base_model, test_loader, device=device)
    rt = evaluate(rg_model, train_eval, device=device, max_batches=cfg.train_eval_max_batches)
    rv = evaluate(rg_model, test_loader, device=device)
    perf += [
        dict(epoch=epoch, run=BASE, train_loss=bt["loss"], train_acc=bt["acc"],
             test_loss=bv["loss"], test_acc=bv["acc"]),
        dict(epoch=epoch, run=RG, train_loss=rt["loss"], train_acc=rt["acc"],
             test_loss=rv["loss"], test_acc=rv["acc"]),
    ]
    bck = _measure(base_model, run_label=BASE, epoch=epoch, global_step=global_step, config=cfg)
    rck = _measure(rg_model, run_label=RG, epoch=epoch, global_step=global_step, config=cfg)
    ww_frames += [bck.metrics, rck.metrics]
    rg_opt.set_supports(rck.supports)
    print(f"epoch={epoch:03d} | SGD={bv['acc']:.4f} | TraceLogRG={rv['acc']:.4f} | supports={rg_opt.get_supports()}")

performance = pd.DataFrame(perf)
weightwatcher = pd.concat(ww_frames, ignore_index=True)
rg_steps = pd.concat(step_frames, ignore_index=True) if step_frames else pd.DataFrame()
corrections = _summarize_corrections(rg_steps)
out = ROOT / "results_sgd_momentum_trace_log_every_step_all_layers_20_epochs"
out.mkdir(parents=True, exist_ok=True)
performance.to_csv(out / "performance_history.csv", index=False)
weightwatcher.to_csv(out / "weightwatcher_rg_history.csv", index=False)
rg_steps.to_csv(out / "rg_step_history.csv", index=False)
corrections.to_csv(out / "rg_correction_summary.csv", index=False)
(out / "experiment_config.json").write_text(json.dumps({
    **cfg.__dict__, "momentum": MOMENTUM, "dampening": DAMPENING, "nesterov": NESTEROV,
}, indent=2))
torch.save({"baseline_model": base_model.state_dict(), "rg_model": rg_model.state_dict(),
            "baseline_optimizer": base_opt.state_dict(), "rg_optimizer": rg_opt.state_dict()},
           out / "final_states.pt")
print("device:", device, "| saved:", out.resolve())

In [ ]:
need = {"run","epoch","layer_name","status","alpha","detX_num","num_pl_spikes",
        "ERG_gap","m_midpoint","trace_log_midpoint_per_eval",
        "trace_boundary_source","ERG_gap_source"}
if need - set(weightwatcher.columns):
    raise RuntimeError(f"Missing original metrics: {sorted(need-set(weightwatcher.columns))}")
w = weightwatcher.loc[weightwatcher.status.eq("ok")].copy()
w["layer"] = w.layer_name.astype(str).str.split(".").str[-1]
for c in ["alpha","detX_num","num_pl_spikes","ERG_gap","m_midpoint","trace_log_midpoint_per_eval"]:
    w[c] = pd.to_numeric(w[c], errors="coerce")
if not w.trace_boundary_source.astype(str).eq("WeightWatcher").all():
    raise RuntimeError("WeightWatcher detX boundary missing; fallback refused.")
if not w.ERG_gap_source.astype(str).eq("WeightWatcher").all():
    raise RuntimeError("The ERG gap is not WeightWatcher's original full-M gap.")
mid = np.floor((w.detX_num + w.num_pl_spikes) / 2).astype(int)
if not np.array_equal(mid.to_numpy(), w.m_midpoint.astype(int).to_numpy()):
    raise RuntimeError("Working rank is not the original PL/detX midpoint.")

display(w[["run","epoch","layer","alpha","detX_num","num_pl_spikes",
           "ERG_gap","m_midpoint","trace_log_midpoint_per_eval"]].tail(30))

fig, ax = plt.subplots(figsize=(10,5))
for run, g in performance.groupby("run"):
    g = g.sort_values("epoch")
    ax.plot(g.epoch, g.test_acc, marker="o", label=f"{run}: test")
    ax.plot(g.epoch, g.train_acc, linestyle="--", label=f"{run}: train")
ax.set(xlabel="Epoch", ylabel="Accuracy", title="Original TraceLogRG on SGD + momentum")
ax.grid(True, alpha=.3); ax.legend(); plt.show()

for metric, ylabel, reference in [
    ("alpha", "WeightWatcher alpha", 2.0),
    ("ERG_gap", "WeightWatcher full-M ERG gap", 0.0),
    ("trace_log_midpoint_per_eval", "Midpoint trace-log per eigenvalue", 0.0),
]:
    for layer, lf in w.groupby("layer"):
        fig, ax = plt.subplots(figsize=(10,5))
        for run, g in lf.groupby("run"):
            g = g.sort_values("epoch")
            ax.plot(g.epoch, g[metric], marker="o", label=run)
        ax.axhline(reference, linestyle="--")
        ax.set(xlabel="Epoch", ylabel=ylabel, title=f"{layer.upper()}: {ylabel}")
        ax.grid(True, alpha=.3); ax.legend(); plt.show()

In [ ]:
# Correction coverage and the primary FC1 falsification table.
if rg_steps.empty:
    raise RuntimeError("No step-level TraceLogRG records were produced.")
s = rg_steps.copy()
s["layer"] = s.parameter.astype(str).str.replace(".weight","",regex=False).str.split(".").str[-1]
for c in ["correction_ratio","base_trace_log_drift","corrected_trace_log_drift"]:
    s[c] = pd.to_numeric(s[c], errors="coerce")
s["fired"] = s.status.eq("ok")
summary = s.groupby(["epoch","layer"], as_index=False).agg(
    opportunities=("global_step","size"), steps=("global_step","nunique"),
    fired=("fired","sum"), fired_fraction=("fired","mean"),
    mean_ratio=("correction_ratio","mean"), max_ratio=("correction_ratio","max"),
    base_drift=("base_trace_log_drift", lambda x: x.abs().sum()),
    corrected_drift=("corrected_trace_log_drift", lambda x: x.abs().sum()),
)
summary["coverage"] = summary.steps / len(train_loader)
summary["drift_residual"] = np.where(summary.base_drift > 0,
                                     summary.corrected_drift / summary.base_drift, np.nan)
display(summary.tail(60))

fc1 = w.loc[w.layer.str.casefold().eq("fc1")].sort_values(["run","epoch"])
spectral = fc1.groupby("run").agg(
    final_alpha=("alpha","last"), minimum_alpha=("alpha","min"),
    mean_abs_alpha_minus_2=("alpha", lambda x: (x-2).abs().mean()),
    final_erg_gap=("ERG_gap","last"),
).reset_index()
accuracy = performance.sort_values("epoch").groupby("run").agg(
    final_test_acc=("test_acc","last"), peak_test_acc=("test_acc","max"),
).reset_index()
display(spectral.merge(accuracy, on="run"))

for metric, ylabel in [
    ("mean_ratio","Mean correction norm / completed SGD step norm"),
    ("fired_fraction","Fraction of steps corrected"),
    ("drift_residual","Corrected / base absolute trace-log drift"),
]:
    fig, ax = plt.subplots(figsize=(10,5))
    for layer, g in summary.groupby("layer"):
        g = g.sort_values("epoch")
        ax.plot(g.epoch, g[metric], marker="o", label=layer.upper())
    ax.set(xlabel="Epoch", ylabel=ylabel, title=metric.replace("_"," ").title())
    ax.grid(True, alpha=.3); ax.legend(); plt.show()

## Decision rule

This is a falsification test of the original paper-based trace-log direction. It succeeds only if the wrapped FC1 trajectory stays closer to $\alpha=2$, the measured contracting trace-log drift is actually removed, and test accuracy is preserved. If FC1 still falls deeply below two while `corrected_trace_log_drift` is driven toward zero, then the original trace-log normal is not sufficient to prevent that SGD-plus-momentum overfitting trajectory.